# SUD LS09 – Klassenkonzept, Kapselung, static

**Quellen:** *LS09A Klassenkonzept* (22 Folien), *LS09B static – nicht static* (8 Folien)

**Ausgangslage (LS09A, Folie 1):** Die Aix-Car-Group wächst von 20 auf 3.000 Fahrzeuge an 10 Standorten. Du programmierst einen Prototyp der KFZ-Verwaltungssoftware.

**Aufbau:** Aufgabe → Infos → Codeblock.
Die Aufgaben sind aus dem Gesprächsprotokoll (LS09A) und der Ausgangssituation (LS09B) abgeleitet. Die Folien enthalten keine nummerierte Aufgabenliste.

**Hinweis zu Jupyter (jjava):** Eine Klasse muss komplett in **einer** Zelle stehen. Änderst du die Klasse, führe ihre Zelle erneut aus und danach die Testzellen.

# Aufgabe 1 – Klassendiagramm `Fahrzeug` entwerfen

## Infos zu der Aufgabe

**Anforderungen aus dem Gesprächsprotokoll vom 17.11.2025:**

- **Speichern:** Kennzeichen, Tankgröße, Tankinhalt, aktuelle Geschwindigkeit, maximale Geschwindigkeit, Motor an/aus
- **Können:** bremsen, beschleunigen, fahren (die Kilometer werden übergeben), tanken
- **Ausgabe:** ein String mit den unveränderlichen Informationen (**Stammdaten**)

**Regeln für das UML-Klassendiagramm (Folien 5–10):**

| Regel | Bedeutung |
|---|---|
| Klassenname mit Großbuchstaben | `Katze`, `Mitarbeiter` |
| 1. Abschnitt: Eigenschaften (Instanzvariablen) | was ein Objekt beschreibt |
| 2. Abschnitt: Methoden | erkennt man an `()` |
| `-` = `private`, `+` = `public` | Kapselung: Eigenschaften immer `-` |
| Datentyp mit `:` anhängen | `- name : String` |
| Rückgabetyp hinter die Klammer | `+ getName() : String` |
| Parameter in die Klammer | `+ setName(name : String)` |
| Get/Set dürfen weggelassen werden | sind „selbstverständlich“ (Folie 10) |

**Beispiel aus Folie 5 (Katze):**

```
┌──────────────────────┐
│        Katze         │
├──────────────────────┤
│ - name : String      │
│ - rasse : String     │
│ - alter : int        │
├──────────────────────┤
│ + laufen()           │
│ + fressen()          │
│ + jagen()            │
└──────────────────────┘
```

**Was du dir überlegen musst:**

- Welcher Datentyp passt jeweils? (Kennzeichen = Text, Tank = Kommazahl oder ganze Zahl?, Motor = nur zwei Zustände)
- Welche Eigenschaften gehören zu den **Stammdaten**? Stammdaten ändern sich nach dem Anlegen nicht.
- **Offen im Protokoll:** Wie stark ändert `beschleunigen`/`bremsen` die Geschwindigkeit? Wie viel Kraftstoff verbraucht `fahren`? Das ist nicht festgelegt. Triff eine Annahme und schreib sie dazu, oder kläre es mit der Lehrkraft.

In [59]:
// Aufgabe 1 – Klassendiagramm als Kommentar
//
// Annahmen (Verbrauch, Schrittweite beim Beschleunigen ...):
// -

public class Car{

    private String plate;
    private double fuelLevel;
    private boolean isActive;
    private double maxFuelLevel;
    private int currentSpeed = 0;
    private double drivenKM = 0;
    private double consumptionRoad;
    private double consumptionHighWay;
    private static int numberOfCars = 0;
    private int id;
    
    // Format: 1-3 letters (place, umlauts allowed) + space + 1-2 letters + space + 1-4 digits, e.g. "AC XY 1234"
    private static final String PLATE_PATTERN = "[A-ZÄÖÜ]{1,3} [A-Z]{1,2} [0-9]{1,4}";

    public Car(String plate, double fuelLevel, boolean isRunning, double maxFuelLevel, double road, double highway){
        if(!setPlate(plate)){
            throw new IllegalArgumentException("A car needs a valid plate: " + plate);
        }
        this.isActive = isRunning;
        this.maxFuelLevel = maxFuelLevel;
        if(fuelLevel < 0){
            System.out.println("Fuel level can't be negative, set to 0: " + fuelLevel);
            fuelLevel = 0;
        }
        if(fuelLevel > maxFuelLevel){
            System.out.println("Fuel level is above maximum, set to maximum: " + fuelLevel);
            fuelLevel = maxFuelLevel;
        }
        this.fuelLevel = fuelLevel;
        this.consumptionRoad = road;
        this.consumptionHighWay = highway;
        this.numberOfCars+=1;
        id = Car.numberOfCars;
    }

    // static: only checks the given text, needs no object
    public static boolean isValidPlate(String plate){
        return plate != null && plate.matches(PLATE_PATTERN);
    }

    public boolean setPlate(String plate){
        if(plate != null){
            plate = plate.trim().toUpperCase();
        }
        if(!isValidPlate(plate)){
            System.out.println("Invalid plate: " + plate + " (expected format: AC XY 1234)");
            return false;
        }
        this.plate = plate;
        return true;
    }

    public String getPlate(){return this.plate;}
    public double getFuelLevel(){return this.fuelLevel;}
    public boolean getMotorStatus(){return this.isActive;}
    public boolean toggleMotor(){
        this.isActive = !this.isActive;
        return isActive;
    }
    // Adds the given liters. Everything above the tank size overflows
    public double refuel(double liters){
        if(liters <= 0){
            System.out.println("Invalid amount of liters: " + liters);
            return this.fuelLevel;
        }
        if(this.isActive){
            System.out.println("You cannot refuel while the motor is still on");
            return this.fuelLevel;
        }
        double newLevel = this.fuelLevel + liters;
        if(newLevel > this.maxFuelLevel){
            System.out.println("Tank is full, " + (newLevel - this.maxFuelLevel) + " l overflowed");
            newLevel = this.maxFuelLevel;
        }
        this.fuelLevel = newLevel;
        return this.fuelLevel;
    }

    public int eccelerate(int speed){
        if(!this.isActive){
            System.out.println("you cant accelerate with an non started motor");
        }
        if((this.currentSpeed+=speed) < 0){
            this.currentSpeed = 0;
            return this.currentSpeed;
        }
        this.currentSpeed += speed;
        return this.currentSpeed;
    }

    public int deccelerate(int speed){
        return eccelerate((speed*-1));
    }

    public double driveOnRoad(double kilometre){
        double expectedConsumption = kilometre * this.consumptionRoad;
        if(this.fuelLevel - expectedConsumption <= 0){
            System.out.println("The input range can not be reached");
            return this.drivenKM;
        }
        this.drivenKM += kilometre;
        return this.drivenKM;
    }

    public double driveOnHighWay(double kilometre){
        double expectedConsumption = (kilometre * this.consumptionHighWay)/100;
        if(this.fuelLevel - expectedConsumption <= 0){
            System.out.println("The input range can not be reached");
            return this.drivenKM;
        }
        this.fuelLevel -= expectedConsumption;
        this.drivenKM += kilometre;
        return this.drivenKM;
    }

    public String toString(){
        return "Displaying infos of car: " + this.plate + ": \n Max fuel Level: " + this.maxFuelLevel +"\n Road Cosumption: " + this.consumptionRoad + "\n Highway Consumption: " + this.consumptionHighWay;
    }

    public static int getNumberOfCars(){return Car.numberOfCars;}
    public int getId(){return this.id;}
    
}

# Aufgabe 2 – Klasse `Fahrzeug` mit Kapselung (Eigenschaften, Getter, Setter)

## Infos zu der Aufgabe

**Kapselung (Folie 7, 14–16):** Instanzvariablen sind immer `private`. Zugriff nur über `public` Methoden. Nur so kann die Klasse ungültige Werte abfangen.

**`this` (Folie 11):** `this.name` ist die Eigenschaft des Objekts, `name` allein ist der Parameter.

**Lebensdauer (Folie 17):** Der Parameter „stirbt“ am Ende der Methode, die Eigenschaft lebt im Objekt weiter.

**Setter prüfen Werte (Folie 15/16):** Ein Setter korrigiert oder verwirft ungültige Werte. Er kennt aber nur **sein** Objekt, nicht alle anderen.

**Beispiel (andere Klasse – Lampe):**

```java
public class Lampe {
    private int helligkeit;      // 0 bis 100
    private boolean an;

    public void setHelligkeit(int helligkeit) {
        if (helligkeit < 0)   helligkeit = 0;     // nach unten begrenzen
        if (helligkeit > 100) helligkeit = 100;   // nach oben begrenzen
        this.helligkeit = helligkeit;
    }
    public int getHelligkeit() { return this.helligkeit; }

    public boolean isAn() { return this.an; }    // boolean-Getter heißen oft isXxx()
}
```

**Was du dir überlegen musst:**

- Welche Werte sind für Tankgröße, Tankinhalt und Geschwindigkeiten ungültig? (negativ? größer als ein Maximum?)
- Der Tankinhalt darf nie größer als die Tankgröße sein. Die aktuelle Geschwindigkeit nie größer als die maximale.
- Braucht jede Eigenschaft einen **öffentlichen** Setter? Denk an die Stammdaten und an den Motor: Soll man den Tankinhalt direkt setzen können oder nur über `tanken`?

In [34]:
// Aufgabe 2 – Klasse Fahrzeug
// Aufgabe 3 und 4 ergänzt du ebenfalls in dieser Zelle und führst sie danach erneut aus.

public class Fahrzeug {

    // TODO: private Eigenschaften

    // TODO: Getter und Setter mit Prüfung

}

# Aufgabe 3 – Methoden `tanken`, `beschleunigen`, `bremsen`, `fahren`

## Infos zu der Aufgabe

**Instanzmethoden (Folie 20, Vokabeln Folie 21):** Normale Methoden (ohne `static`) arbeiten mit den Instanzvariablen **eines** Objekts. Deshalb ruft man sie über eine Instanz auf: `objekt.methode()`.

**Gute Praxis:** Methoden, die den Zustand ändern, nutzen intern die eigenen Setter oder prüfen selbst. So gibt es die Prüfregel nur an einer Stelle.

**Beispiel (andere Klasse – Konto):**

```java
public class Konto {
    private double stand;
    private double limit = 500;

    public void einzahlen(double betrag) {
        if (betrag <= 0) return;                 // ungültig → nichts tun
        this.stand = this.stand + betrag;
    }

    public boolean abheben(double betrag) {      // Rückgabe: hat es geklappt?
        if (betrag <= 0 || betrag > this.stand + this.limit) return false;
        this.stand = this.stand - betrag;
        return true;
    }
}
```

**Was du dir überlegen musst:**

| Methode | Fragen |
|---|---|
| `tanken` | Wie viel wird übergeben? Was passiert, wenn mehr getankt wird, als in den Tank passt? Darf man mit laufendem Motor tanken? |
| `beschleunigen` | Um wie viel? Was passiert an der Höchstgeschwindigkeit? Geht das bei Motor aus? |
| `bremsen` | Um wie viel? Die Geschwindigkeit darf nicht negativ werden. |
| `fahren(km)` | Die Kilometer werden übergeben. Verbrauch pro km ist **nicht festgelegt** → Annahme treffen. Was, wenn der Tank vorher leer ist? |
| Motor | Wie wird der Motor an- und ausgeschaltet? Das Protokoll nennt dafür keine Methode, die Eigenschaft gibt es aber. |

In [35]:
// Aufgabe 3 – Methoden in der Klasse aus Aufgabe 2 ergänzen, Zelle dort neu ausführen.
// Hier die Methoden testen:

// TODO: Fahrzeug-Instanz anlegen

// TODO: tanken testen (normal, zu viel, negativ)

// TODO: beschleunigen / bremsen testen (über Maximum, unter 0, Motor aus)

// TODO: fahren testen (genug Kraftstoff, zu wenig Kraftstoff)

# Aufgabe 4 – `toString()` für die Stammdaten

## Infos zu der Aufgabe

**`toString()` (Folie 11):** Gibt einen String zurück, der das Objekt beschreibt. `System.out.println(objekt)` ruft `toString()` automatisch auf.

**Laut Protokoll:** nur die **unveränderlichen** Informationen (Stammdaten), nicht den aktuellen Zustand.

**Beispiel (andere Klasse – Buch):**

```java
public class Buch {
    private String isbn;
    private String titel;
    private int ausgeliehenAnzahl;   // ändert sich → kein Stammdatum

    public String toString() {
        return "ISBN: " + this.isbn + " | Titel: " + this.titel;
    }
}

Buch b = new Buch();
System.out.println(b);          // gleiche Ausgabe wie b.toString()
```

**Was du dir überlegen musst:** Welche deiner Eigenschaften aus Aufgabe 1 ändern sich nie? Nur diese gehören in den String.

In [57]:
// Aufgabe 4 – toString() in der Klasse aus Aufgabe 2 ergänzen, Zelle dort neu ausführen.
// Hier testen:
Car testCar = new Car("DN NN 1704", 40, false, 50, 4.7, 3.0);
System.out.println(testCar);
// TODO: Instanz anlegen und mit System.out.println ausgeben

// --- Plate filter ---
System.out.println("Soll true  | Ist " + Car.isValidPlate("AC XY 1234"));
System.out.println("Soll true  | Ist " + Car.isValidPlate("K A 1"));
System.out.println("Soll true  | Ist " + Car.isValidPlate("LÖ AB 99"));
System.out.println("Soll false | Ist " + Car.isValidPlate("DNN1704"));      // no spaces
System.out.println("Soll false | Ist " + Car.isValidPlate("ABCD X 1"));     // 4 letters place
System.out.println("Soll false | Ist " + Car.isValidPlate("AC XYZ 1"));     // 3 letters middle
System.out.println("Soll false | Ist " + Car.isValidPlate("AC XY 12345"));  // 5 digits
System.out.println("Soll false | Ist " + Car.isValidPlate(null));
System.out.println("Soll false | Ist " + testCar.setPlate("AC 1234") + " | plate stays " + testCar.getPlate());
System.out.println("Soll true  | Ist " + testCar.setPlate(" ac xy 12 ") + " | plate now " + testCar.getPlate());
try {
    new Car("DNN1704", 10, false, 50, 4.7, 3.0);
    System.out.println("FEHLER: invalid plate accepted");
} catch (IllegalArgumentException e) {
    System.out.println("OK, constructor rejected: " + e.getMessage());
}

// --- Refuel filter (tank 50 l, currently 40 l) ---
System.out.println("Soll 45.0 | Ist " + testCar.refuel(5));
System.out.println("Soll 50.0 | Ist " + testCar.refuel(20));   // 15 l overflow
System.out.println("Soll 50.0 | Ist " + testCar.refuel(-3));   // invalid
testCar.toggleMotor();
System.out.println("Soll 50.0 | Ist " + testCar.refuel(1));    // motor on
testCar.toggleMotor();
Car overfull = new Car("AC X 1", 80, false, 50, 4.7, 3.0);
System.out.println("Soll 50.0 | Ist " + overfull.getFuelLevel());


Displaying infos of car: DN NN 1704: 
 Max fuel Level: 50.0
 Road Cosumption: 4.7
 Highway Consumption: 3.0
Soll true  | Ist true
Soll true  | Ist true
Soll true  | Ist true
Soll false | Ist false
Soll false | Ist false
Soll false | Ist false
Soll false | Ist false
Soll false | Ist false
Invalid plate: AC 1234 (expected format: AC XY 1234)
Soll false | Ist false | plate stays DN NN 1704
Soll true  | Ist true | plate now AC XY 12
Invalid plate: DNN1704 (expected format: AC XY 1234)
OK, constructor rejected: A car needs a valid plate: DNN1704
Soll 45.0 | Ist 45.0
Tank is full, 15.0 l overflowed
Soll 50.0 | Ist 50.0
Invalid amount of liters: -3.0
Soll 50.0 | Ist 50.0
You cannot refuel while the motor is still on
Soll 50.0 | Ist 50.0
Fuel level is above maximum, set to maximum: 80.0
Soll 50.0 | Ist 50.0


# Aufgabe 5 – Instanzen erzeugen und alle Methoden testen

## Infos zu der Aufgabe

**Deklaration und Erzeugung (Folie 12/13):**

| Schritt | Code | Was passiert |
|---|---|---|
| Deklaration | `Lampe flur;` | nur die **Referenz** wird angelegt, noch kein Objekt |
| Erzeugung | `flur = new Lampe();` | erst `new` erzeugt die **Instanz** |
| beides | `Lampe kueche = new Lampe();` | in einer Zeile |
| Zugriff | `kueche.setHelligkeit(80);` | **Punktoperator** auf `public` Elemente |

**Testen ist Pflicht (Folie 13):** Zu jeder Klasse gehört der Test **aller** Methoden, auch Getter, Setter und `toString()`.

**Kapselung prüfen (Folie 14):** Schreib testweise einen direkten Zugriff wie `kueche.helligkeit = 500;`. Mit `private` muss das einen Fehler geben. Danach die Zeile wieder löschen.

**In Eclipse (Folie 22)** wären das zwei Klassen: `Fahrzeug` und `FahrzeugMain` mit `main`. In Jupyter schreibst du die Testaufrufe direkt in die Zelle.

**Tipp für ein Testprotokoll:** Gib bei jedem Test Soll- und Ist-Wert aus, z. B. `System.out.println("Soll: 100 | Ist: " + kueche.getHelligkeit());` – wie beim Black-Box-Test aus LS04.

In [47]:
//s.o

# Aufgabe 6 – `static`-Eigenschaft: Fahrzeuge zählen und automatische ID

## Infos zu der Aufgabe

**Idee (LS09B, Folie 3/4):** IDs sollen automatisch hochzählen, damit es keine Dopplungen gibt. Die Anzahl ist keine Eigenschaft **eines** Objekts, sondern **eine Eigenschaft für alle** Objekte.

**static – nicht static (LS09B, Folie 5/6):**

| | static | nicht static |
|---|---|---|
| Eigenschaft | **Klassenvariable**, gilt für alle Objekte, existiert einmal | **Instanzvariable**, beschreibt ein einzelnes Objekt |
| Methode | arbeitet **ohne** Instanzvariablen | braucht Instanzvariablen eines Objekts |
| Aufruf | über die Klasse: `Klasse.methode()` | über die Instanz: `objekt.methode()` |
| UML | **unterstrichen** | normal |

**Regel (Folie 7/8):** `static` Methoden dürfen nur auf `static` Eigenschaften zugreifen. Kein Setter für den Zähler, weil nur der Konstruktor hochzählt.

**Konstruktor (Ergänzung, auf Folie 8 verwendet):** Eine Methode mit dem Namen der Klasse und **ohne** Rückgabetyp. Sie läuft automatisch bei `new`.

**Beispiel (andere Klasse – Ticket):**

```java
public class Ticket {
    private static int ausgegeben = 0;   // einmal für ALLE Tickets
    private int nummer;                  // pro Ticket

    public Ticket() {                    // Konstruktor
        ausgegeben++;
        this.nummer = 500 + ausgegeben;
    }
    public int getNummer() { return this.nummer; }
    public static int getAusgegeben() { return ausgegeben; }
}

Ticket t1 = new Ticket();
Ticket t2 = new Ticket();
System.out.println(t2.getNummer());           // 502
System.out.println(Ticket.getAusgegeben());   // 2 – Aufruf über die Klasse
```

**Was du dir überlegen musst:**

- Bekommt `Fahrzeug` eine eigene ID zusätzlich zum Kennzeichen? (Kennzeichen kann sich z. B. bei Umzug ändern.)
- Welche Werte übergibst du dem Konstruktor? Nutze darin die Setter aus Aufgabe 2, damit die Prüfung greift.
- Ändere dein Klassendiagramm aus Aufgabe 1: Zähler und `static` Getter unterstreichen (im Kommentar z. B. mit `_` markieren).

In [60]:
// Aufgabe 6 – Tests: static Zähler und automatische ID
// Erwartet in Car:  public int getId()   und   public static int getNumberOfCars()
// Die Tests rechnen relativ zum Zählerstand vor dem Test, weil jede ausgeführte Testzelle neue Autos anlegt.

void check(String test, Object soll, Object ist) {
    String ergebnis = soll.equals(ist) ? "OK    " : "FEHLER";
    System.out.println(ergebnis + " | " + test + " | Soll: " + soll + " | Ist: " + ist);
}

// 1. Zählerstand vor dem Test merken (Aufruf über die KLASSE)
int start = Car.getNumberOfCars();
System.out.println("Zählerstand vor dem Test: " + start);

// 2. Drei Autos anlegen -> Zähler +3
Car a = new Car("AC A 1", 10, false, 50, 4.7, 3.0);
Car b = new Car("AC B 2", 10, false, 50, 4.7, 3.0);
Car c = new Car("AC C 3", 10, false, 50, 4.7, 3.0);
check("Zähler nach 3 Autos", start + 3, Car.getNumberOfCars());

// 3. IDs fortlaufend in Reihenfolge der Erzeugung
check("ID Auto a", start + 1, a.getId());
check("ID Auto b", start + 2, b.getId());
check("ID Auto c", start + 3, c.getId());

// 4. Keine Dopplung
check("IDs a/b/c verschieden", true, a.getId() != b.getId() && b.getId() != c.getId() && a.getId() != c.getId());

// 5. static = eine Variable für alle: neues Auto erhöht den Zähler,
//    die IDs der bestehenden Autos bleiben gleich
Car d = new Car("AC D 4", 10, false, 50, 4.7, 3.0);
check("Zähler nach 4. Auto", start + 4, Car.getNumberOfCars());
check("ID Auto a unverändert", start + 1, a.getId());
check("ID Auto d", start + 4, d.getId());

// 6. Ungültiges Kennzeichen: Konstruktor bricht ab -> Zähler darf NICHT steigen
try {
    new Car("UNGUELTIG", 10, false, 50, 4.7, 3.0);
} catch (IllegalArgumentException e) {
    System.out.println("       (Konstruktor hat abgelehnt: " + e.getMessage() + ")");
}
check("Zähler nach abgelehntem Auto", start + 4, Car.getNumberOfCars());

// 7. Keine Lücke in den IDs nach dem abgelehnten Auto
Car e = new Car("AC E 5", 10, false, 50, 4.7, 3.0);
check("ID nach abgelehntem Auto", start + 5, e.getId());

// 8. ID ändert sich nicht durch andere Methoden
a.setPlate("AC Z 99");
a.refuel(5);
check("ID a nach setPlate/refuel", start + 1, a.getId());

Zählerstand vor dem Test: 0
OK     | Zähler nach 3 Autos | Soll: 3 | Ist: 3
OK     | ID Auto a | Soll: 1 | Ist: 1
OK     | ID Auto b | Soll: 2 | Ist: 2
OK     | ID Auto c | Soll: 3 | Ist: 3
OK     | IDs a/b/c verschieden | Soll: true | Ist: true
OK     | Zähler nach 4. Auto | Soll: 4 | Ist: 4
OK     | ID Auto a unverändert | Soll: 1 | Ist: 1
OK     | ID Auto d | Soll: 4 | Ist: 4
Invalid plate: UNGUELTIG (expected format: AC XY 1234)
       (Konstruktor hat abgelehnt: A car needs a valid plate: UNGUELTIG)
OK     | Zähler nach abgelehntem Auto | Soll: 4 | Ist: 4
OK     | ID nach abgelehntem Auto | Soll: 5 | Ist: 5
OK     | ID a nach setPlate/refuel | Soll: 1 | Ist: 1


# Aufgabe 7 – Refactoring Dreieck: `static` Prüfmethode

## Infos zu der Aufgabe

**Ausgangssituation (LS09B, Folie 2):** Bei AixCuisine werden Dreiecke durch fehlerhafte Änderungen der Seiten nicht konstruierbar. Danach liefert die Software unsinnige Ergebnisse.

**Auftrag aus dem Design-Meeting:** Ein Fehlerabfang für nicht konstruierbare Dreiecke, der **im Konstruktor und in den drei Settern** benutzt werden kann.

**Refactoring:** Code verbessern, ohne neue Funktionalität hinzuzufügen.

**Dreiecksungleichung (Ergänzung):** Ein Dreieck mit den Seiten a, b, c ist konstruierbar, wenn alle Seiten > 0 sind und jede Seite kleiner ist als die Summe der beiden anderen:
`a < b + c` **und** `b < a + c` **und** `c < a + b`

**Warum `static`?** Die Prüfung braucht nur die drei übergebenen Zahlen, keine Instanzvariablen. Sie lässt sich so auch aufrufen, **bevor** ein Objekt existiert.

**Beispiel (andere Klasse – Datum):**

```java
public class Datum {
    private int tag, monat;

    public static boolean istGueltig(int tag, int monat) {   // nur Parameter, kein this
        return monat >= 1 && monat <= 12 && tag >= 1 && tag <= 31;
    }

    public Datum(int tag, int monat) {
        if (istGueltig(tag, monat)) { this.tag = tag; this.monat = monat; }
        else                        { this.tag = 1;   this.monat = 1; }
    }

    public void setMonat(int monat) {
        if (istGueltig(this.tag, monat)) this.monat = monat;      // alter Wert bleibt sonst
    }
}

System.out.println(Datum.istGueltig(31, 13));   // false – ohne Objekt aufrufbar
```

**Was du dir überlegen musst:**

- In einem Setter ändert sich nur **eine** Seite. Welche drei Werte übergibst du der Prüfmethode?
- Was passiert, wenn die Prüfung fehlschlägt? Alten Wert behalten, Meldung ausgeben, Standardwerte?
- Deine Dreieck-Klasse aus der vorherigen Lernsituation liegt nicht in deinen Ordnern. Bau sie hier neu oder kopiere sie hinein.

**Hinweis:** Die weiteren Punkte aus dem Meeting (Container für Fuhrpark/Mitarbeiter, Sortierung) sind hier nicht als Aufgabe enthalten. Die Folien in LS09 erklären dazu noch nichts.

In [64]:
// Aufgabe 7 – Dreieck mit static Prüfmethode

public class Dreieck {

    // TODO: private Seiten
    private double sideA;
    private double sideB;
    private double sideC;
    // TODO: static Prüfmethode (Dreiecksungleichung)
    private static boolean checkPosibility(double sideA, double sideB, double sideC){
        if(sideA < (sideB + sideC))return false;
        if(sideC < (sideB + sideA))return false;
        if(sideB < (sideA + sideC))return false;
        return true;
    }
    // TODO: Konstruktor nutzt die Prüfmethode
    public Dreieck(double sideA, double sideB, double sideC){
        this.sideA = sideA;
        this.sideB = sideB;
        this.sideC = sideC;
    }
    // TODO: drei Setter nutzen die Prüfmethode
    public double setSideA(double sideA){
        if(!checkPosibility(this.sideA, this.sideB, this.sideC)){
            return this.sideA;
        }
        this.sideA = sideA;
        return this.sideA;
    }

    public double setSideB(double sideB){
        if(!checkPosibility(this.sideA, this.sideB, this.sideC)){
            return this.sideB;
        }
        this.sideB = sideB;
        return this.sideB;
    }

    public double setSideC(double sideC){
        if(!checkPosibility(this.sideA, this.sideB, this.sideC)){
            return this.sideC;
        }
        this.sideC = sideC;
        return this.sideC;
    }
    
}

// TODO: testen – konstruierbar, nicht konstruierbar, Seite 0 oder negativ,
//       gültiges Dreieck per Setter ungültig machen

In [65]:
// Aufgabe 7 – Tests: Dreieck mit static Prüfmethode
// Erwartet in Dreieck:  getSideA(), getSideB(), getSideC()  (Rückgabe double)
// Konstruktor bei nicht konstruierbarem Dreieck: entweder IllegalArgumentException werfen
// oder die ungültigen Seiten NICHT übernehmen – beides zählt hier als "abgelehnt".

void check(String test, Object soll, Object ist) {
    String ergebnis = soll.equals(ist) ? "OK    " : "FEHLER";
    System.out.println(ergebnis + " | " + test + " | Soll: " + soll + " | Ist: " + ist);
}

boolean wirdAbgelehnt(double a, double b, double c) {
    try {
        Dreieck d = new Dreieck(a, b, c);
        // keine Exception: dann dürfen die ungültigen Seiten nicht gespeichert sein
        return !(d.getSideA() == a && d.getSideB() == b && d.getSideC() == c);
    } catch (IllegalArgumentException ex) {
        return true;
    }
}

// 1. Gültige Dreiecke werden angelegt und die Seiten gespeichert
Dreieck t1 = new Dreieck(3, 4, 5);
check("3-4-5: Seite a", 3.0, t1.getSideA());
check("3-4-5: Seite b", 4.0, t1.getSideB());
check("3-4-5: Seite c", 5.0, t1.getSideC());
Dreieck t2 = new Dreieck(2, 2, 2);
check("gleichseitig 2-2-2: Seite a", 2.0, t2.getSideA());

// 2. Konstruktor: jede der drei Ungleichungen einzeln verletzen
check("Konstruktor 10-1-2 (a zu lang)", true, wirdAbgelehnt(10, 1, 2));
check("Konstruktor 1-10-2 (b zu lang)", true, wirdAbgelehnt(1, 10, 2));
check("Konstruktor 1-2-10 (c zu lang)", true, wirdAbgelehnt(1, 2, 10));

// 3. Grenzfall: a + b == c ist eine Linie, kein Dreieck
check("Konstruktor 1-2-3 (flach)", true, wirdAbgelehnt(1, 2, 3));

// 4. Seite 0 oder negativ
check("Konstruktor 0-4-5", true, wirdAbgelehnt(0, 4, 5));
check("Konstruktor 3-(-4)-5", true, wirdAbgelehnt(3, -4, 5));

// 5. Setter mit gültigem Wert -> wird übernommen
t1.setSideC(6);
check("setSideC(6) bei 3-4-?", 6.0, t1.getSideC());

// 6. Setter mit Wert, der das Dreieck unmöglich macht -> alter Wert bleibt
t1.setSideA(20);
check("setSideA(20) abgelehnt", 3.0, t1.getSideA());
t1.setSideB(10);
check("setSideB(10) abgelehnt (3+6 < 10)", 4.0, t1.getSideB());
t1.setSideC(7);
check("setSideC(7) abgelehnt (3+4 = 7, flach)", 6.0, t1.getSideC());

// 7. Setter mit 0 oder negativ -> alter Wert bleibt
t1.setSideB(0);
check("setSideB(0) abgelehnt", 4.0, t1.getSideB());
t1.setSideA(-3);
check("setSideA(-3) abgelehnt", 3.0, t1.getSideA());

// 8. Seiten sind Instanzvariablen: Änderung an t1 ändert t2 nicht
t1.setSideA(4);
check("t1 a geändert auf 4", 4.0, t1.getSideA());
check("t2 a unverändert", 2.0, t2.getSideA());

CompilationException: [snippet: [check("3-4-5: Seite a", 3.0, t1.getSideA());], status: REJECTED]